# 04 — First measured improvement: common-word removal

**Question:** do common function words distract BM25 from the clause the user needs?

At baseline k=5, 103 of 140 train questions find their contract but miss the answer span entirely. We will change only token filtering, keeping chunk size, overlap, BM25 parameters, threshold, and k fixed.

The hypothesis, primary objective, and guardrails are declared in `experiments/OBJECTIVE.md` and `experiments/objective.json` before candidate measurement. This is one iteration; accepted and rejected results are both retained. No LLM or optional retrieval feature is used.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from run_experiment import run_iteration
from retrieval import BM25Retriever, load_inputs, tokenize

objective = json.loads((ROOT / 'experiments/objective.json').read_text(encoding='utf-8'))
display(pd.Series(objective, name='Acceptance policy'))
print('Original:', tokenize('Which clause does not allow sublicensing?'))
print('Filtered:', tokenize('Which clause does not allow sublicensing?', True))

primary_metric                                                      macro_character_recall_at_5
minimum_recall_gain_absolute                                                               0.01
minimum_precision_ratio                                                                    0.95
maximum_false_abstention_increase_absolute                                                 0.02
maximum_correct_abstention_drop_absolute                                                   0.05
latency_p95_multiplier                                                                      2.0
latency_p95_floor_ms                                                                       50.0
comparison                                    candidate versus unchanged baseline for iterat...
evaluation_split                                                                          train
Name: Acceptance policy, dtype: object

Original: ['which', 'clause', 'does', 'not', 'allow', 'sublicensing']
Filtered: ['clause', 'not', 'allow', 'sublicensing']


## Run one iteration

The command-line equivalent is `python run_experiment.py`. The runner evaluates the unchanged baseline, diagnoses train failures, evaluates the fixed candidate, applies the acceptance policy, and records both outcomes.

Each run creates a new result folder and appends a log entry. Rerunning iteration 1 verifies the same proposal; it does not count as another distinct improvement. Timing can vary. Both 140 answerable and 23 unanswerable train questions are included; test questions are never retrieved.

In [2]:
output, reference, candidate, log = run_iteration(ROOT)
print('Saved:', output)
print('Before:', log['diagnosis_before'])
print('After:', log['diagnosis_after'])

Saved: C:\Users\vatsa\OneDrive\Desktop\JioStar\results\iteration_01\20261101T021505909651Z
Before: {'complete_coverage': 15, 'right_document_wrong_section': 103, 'partial_coverage': 22}
After: {'complete_coverage': 13, 'right_document_wrong_section': 107, 'partial_coverage': 20}


## Compare measured results

Recall measures how much of the needed clause we found. Precision measures how much returned text is relevant. Five returned passages represent a fixed reviewer budget. Correct negative abstention is still a weakness of the baseline; a retrieval improvement alone does not solve it.

The 1-point recall target and guardrail tolerances are case-study engineering choices. Train gains are development evidence, not claims of generalization or statistical significance.

In [3]:
tables = []
for label, result in [('baseline', reference), ('stopwords', candidate)]:
    frame = pd.DataFrame(result['answerable']['metrics'])
    frame.insert(0, 'approach', label)
    tables.append(frame)
comparison = pd.concat(tables, ignore_index=True)
display(comparison.style.format({'precision':'{:.2%}', 'recall':'{:.2%}', 'hit':'{:.2%}'}))
display(pd.DataFrame([
    dict(approach=label,
         false_abstention=result['answerable']['answerable_false_abstention_rate'],
         correct_abstention=result['negative']['correct_abstention_rate'],
         p95_ms=result['answerable']['latency_ms']['p95'])
    for label, result in [('baseline', reference), ('stopwords', candidate)]
]))
print('ACCEPTED' if log['decision']['accepted'] else 'REJECTED')
display(pd.Series(log['decision']['checks'], name='Passed'))
comparison.to_csv(output / 'comparison.csv', index=False)

,approach,k,n_queries,precision,recall,hit
0,baseline,1,140,0.56%,1.72%,3.57%
1,baseline,3,140,1.36%,7.43%,12.14%
2,baseline,5,140,1.74%,18.16%,26.43%
3,baseline,10,140,1.81%,32.16%,43.57%
4,stopwords,1,140,0.57%,1.58%,3.57%
5,stopwords,3,140,1.47%,8.14%,12.86%
6,stopwords,5,140,1.61%,15.89%,23.57%
7,stopwords,10,140,1.78%,31.87%,43.57%


,approach,false_abstention,correct_abstention,p95_ms
0,baseline,0.0,0.0,11.981420
1,stopwords,0.0,0.0,8.137755


REJECTED


recall_gain                False
precision                  False
answerable_abstention       True
unanswerable_abstention     True
latency                     True
Name: Passed, dtype: bool

## Try either runnable approach

Edit `question` and select `baseline` or `stopwords`. Both configurations remain available even if this experiment is rejected. Edit a separate config file for future experiments; retain the original baseline and all previous results.

In [4]:
approach = 'stopwords'
_, documents, _ = load_inputs(ROOT)
config_name = {'baseline':'bm25_baseline', 'stopwords':'bm25_stopwords'}[approach]
config = json.loads((ROOT / 'configs' / f'{config_name}.json').read_text(encoding='utf-8'))
model = BM25Retriever(documents, config)
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,42.828320,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,26.705922,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
2,21.596644,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
3,11.581403,cuad/FLOTEKINDUSTRIESINCCN_05_09_2019-EX-10.1-...,1600,2600,"ting Principles, consistently applied), plus a..."
4,10.987919,cuad/KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1...,65600,66600,"grants to Depomed a non-assignable, non-subli..."


## Explain this step

We tested one simple hypothesis: common-word removal might improve clause ranking. We compared both configurations under the same train evaluation and used a predeclared acceptance rule. Keeping rejected ideas prevents presenting only successful experiments.

**Business perspective:** better recall can reduce missed clauses; stable precision limits irrelevant reading. This change adds no API fees or external data transfer, but remains keyword-based and does not establish legal correctness. Local timing is not a production benchmark.

**Checkpoint:** one distinct improvement iteration is complete. At least four further iterations are still required by the brief. We will inspect this result before choosing the next change. Test performance, the working app/CLI, and LLM experiments remain for later steps.

## Recorded outcome: rejected

On the recorded train run, recall@5 changed from **18.16% to 15.89%**, and precision@5 from **1.74% to 1.61%**. The candidate failed both the recall-gain requirement and precision guardrail. Search p95 improved from about **11.98 ms to 8.14 ms**, but speed did not justify reduced clause coverage.

Both approaches still falsely returned passages for all 23 train negatives. The baseline remains selected. Common-word removal changed BM25 weighting, but these measurements do not establish exactly which token changes caused each failure. Inspect individual predictions before making a causal claim.

Interview explanation: “We declared the success criteria first, tested one change, and rejected it when quality dropped. A faster model was not worth missing more contract clauses.”